In [ ]:
import numpy as np
from parmed.amber import AmberParm
from qmhub.units import AMBER_HARTREE_TO_KCAL, AMBER_BOHR_TO_A

In [ ]:
n_windows = 42
nbins = n_windows - 1

prot_name = "WT"
num_prot_atoms = 27086

qm_index2 = np.array([60, 53, 52], dtype=int) - 1 # MM Number to Python
qm_index = np.array([27261,26639,26638], dtype=int) - 1 # MM Number to Python

parm = AmberParm("../input/step3_pbcsetup.parm7")

In [3]:
# Coordinate caches are generated separately:
# pos_all.py builds full pos_all.npy, and qm_pos.py truncates it to qm_pos.npy for analysis.

In [4]:
# got MM atom numbers from step5.00_equilibration.mdout
# grep "QMMM:     1" step5.00_equilibration.mdout -A 67 |awk '{print $3}' | tr '\n' ','

arr = np.array([13767,13768,13769,13770,13771,13772,13779,13780,13781,13782,13783,13784,13785,13786,13787,13788,
         13789,14154,14155,14156,14157,14158,14159,14160,14161,22499,26613,26614,26615,26616,26617,26618,
         26619,26620,26621,26622,26623,26624,26625,26626,26627,26628,26629,26630,26631,26632,26633,26634,
         26635,26636,26637,26638,26639,26640,26641,26642,26643,26644,26645], dtype=int) - 1

#water atoms
#27261,27262,27263,27267,27268,27269,27270,27271,27272]

# total number of non-water atoms minus 1
num_prot_atoms2 = 27146 - 1
resid = np.zeros(num_prot_atoms2, dtype=int)
for i in range(num_prot_atoms2):
    resid[i] = parm.atoms[i].residue.idx


resid = np.delete(resid,arr)
num_prot_res = resid[-1]

res_names = []
for i in np.unique(resid):
    if i <= 1364:
        _res_id = i + 2
        res_names.append("Prot-" + parm.residues[i].name.capitalize() + "%d" % _res_id )
    elif 1365 <= i <= 1368:
        _res_id = i + 2
        res_names.append("MGs-" + parm.residues[i].name.capitalize() + "%d" % _res_id )
    elif 1369 <= i <= 1466:
        _res_id = i + 2
        res_names.append("RNA-" + parm.residues[i].name.capitalize() + "%d" % _res_id )
    elif 1467 <= i <= 1515:
        _res_id = i + 2
        res_names.append("DNA-" + parm.residues[i].name.capitalize() + "%d" % _res_id )

res_names.append('Near')
res_names.append('Far')

np.save('res_names', res_names)

In [47]:
qmmm_forces = []
for i in range(n_windows):
    force = -1 * np.load("../%02d/eda/qmmm_forces.npy" % i).swapaxes(1, 2) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    qmmm_forces.append(force[:, qm_index2])
qmmm_forces = np.array(qmmm_forces)

np.save('qmmm_qm_forces', qmmm_forces)

In [ ]:
qmmm_prot_forces = []
for i in range(n_windows):
    force = -1 * np.load("../%02d/eda/qmmm_prot_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    qmmm_prot_forces.append(force.sum(axis=3))
qmmm_prot_forces = np.array(qmmm_prot_forces)

qmmm_res_forces = []
for i in range(n_windows):
    force = -1 * np.load("../%02d/eda/qmmm_prot_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    nearforce = -1 * np.load("../%02d/eda/qmmm_near_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    farforce = -1 * np.load("../%02d/eda/qmmm_far_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    force2 = np.zeros((250, 3, 3, num_prot_res+2))
    for i in range(num_prot_res):
        force2[:, :, :, i] = force[:, :, :, resid == i].sum(axis=3)
    force2[:,:,:,num_prot_res] = nearforce
    force2[:,:,:,num_prot_res + 1] = farforce
    qmmm_res_forces.append(force2)

qmmm_res_forces = np.array(qmmm_res_forces)

np.save('qmmm_res_forces2', qmmm_res_forces)

In [46]:
gas_forces = []
for i in range(n_windows):
    force = -1 * np.load("../%02d/eda/gas_forces.npy" % i).swapaxes(1, 2) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    gas_forces.append(force[:, qm_index2])
gas_forces = np.array(gas_forces)
np.save('gas_qm_forces.npy', gas_forces)

In [ ]:
res_forces = []
for i in range(n_windows):
    force = -1 * np.load("../%02d/eda/gas_prot_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    nearforce = -1 * np.load("../%02d/eda/gas_near_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    farforce = -1 * np.load("../%02d/eda/gas_far_forces.npy" % i) * (AMBER_HARTREE_TO_KCAL / AMBER_BOHR_TO_A)
    force2 = np.zeros((len(force), 3, 3, num_prot_res+2))
    for i in range(num_prot_res):
        force2[:, :, :, i] = force[:, :, :, resid == i].sum(axis=3)
    force2[:,:,:,num_prot_res] = nearforce
    force2[:,:,:,num_prot_res + 1] = farforce
    res_forces.append(force2)
res_forces = np.array(res_forces)

np.save('gas_res_forces2', res_forces)

`lj_prot_forces.npy` keeps the atom dimension needed for residue decomposition. `lj_prot_forces_sum.npy` is already summed over all protein atoms and is only appropriate for total protein vdW force workflows.

In [52]:
vdw_res_forces = []
for i in range(n_windows):
    force = np.load("../%02d/eda/lj_prot_forces.npy" % i)
    nearforce = np.load("../%02d/eda/lj_near_forces.npy" % i)
    force2 = np.zeros((len(force), 3, 3, num_prot_res+2))
    for i in range(num_prot_res):
        force2[:, :, :, i] = force[:, :, :, resid == i].sum(axis=3)
    force2[:,:,:,num_prot_res] = nearforce
    vdw_res_forces.append(force2)
vdw_res_forces = np.array(vdw_res_forces)

np.save('vdw_res_forces2', vdw_res_forces)